# Fall Detection Pipeline (Nano)

End-to-end real-time pipeline for Jetson Nano:

```
camera/video  ->  MoveNet (TRT FP16)  ->  51-dim features (sliding deque)
              ->  GRU (PyTorch CPU)  ->  fall_prob  ->  alarm logic  ->  display
```

**Run order**: configure (cell 2) -> load models (cell 3) -> define helpers (cell 4)
-> pick a runner cell:
- **cell 5**: live webcam
- **cell 6**: pre-recorded video file
- **cell 7**: quick MoveNet-only sanity check (no GRU)

Stop a running cell with Jupyter's interrupt button (or `q` in the OpenCV window if `DISPLAY=True`).

In [ ]:
# === Config ===========================================================
# Paths
ENGINE_PATH = 'movenet/output/pose_fp16.engine'
GRU_PATH    = 'models/fall_gru.pth'

# Input
CAMERA_INDEX = 0          # used by the live-webcam cell
VIDEO_PATH   = 'data_Le2i/Coffee_room_01/Coffee_room_01/Videos/video (1).avi'
CAM_WIDTH    = 640
CAM_HEIGHT   = 480

# Alarm
PROB_THRESHOLD = 0.6      # per-frame fall_prob cutoff
ALARM_FRAMES   = 8        # consecutive over-threshold frames to fire
ALARM_HOLD     = 50       # frames the alarm stays on after firing

# Inference / display
INFERENCE_STRIDE = 1      # run GRU every N frames; raise if Nano can't keep up
DRAW_KP_THR      = 0.2    # min keypoint score to draw skeleton

DISPLAY = True            # show OpenCV window. Set False for headless / SSH.
SAVE_OUT = None           # e.g. 'demo.mp4' to record annotated output

print('config loaded')

In [ ]:
# === Imports & model loading ==========================================
import os, sys, time, collections
import cv2
import numpy as np
import torch

# project-root imports
sys.path.insert(0, os.getcwd())

from movenet.movenet_trt import MoveNetTRT, draw_keypoints
from features import KeypointFeatureExtractor, FEATURE_DIM
from fall_model import FallDetectionGRU

print('[INFO] Loading MoveNet TRT engine: {}'.format(ENGINE_PATH))
movenet = MoveNetTRT(ENGINE_PATH)

print('[INFO] Loading GRU: {}'.format(GRU_PATH))
gru, ckpt = FallDetectionGRU.load_from(GRU_PATH, map_location='cpu')
gru.eval()
SEQ_LEN = ckpt.get('seq_len', 30)
print('     seq_len    : {}'.format(SEQ_LEN))
print('     params     : {:,}'.format(gru.num_parameters()))
if 'val_metrics' in ckpt:
    m = ckpt['val_metrics']
    print('     val F1     : {:.3f}  (acc={:.3f}, P={:.3f}, R={:.3f})'.format(
        m.get('f1', 0), m.get('acc', 0), m.get('precision', 0), m.get('recall', 0)))

In [ ]:
# === UI overlay + run loop helpers ====================================
def draw_overlay(frame, fall_prob, alarm_active, fps, seq_filled, seq_len):
    """Top alarm banner, bottom fall_prob bar, FPS, buffer status."""
    h, w = frame.shape[:2]

    if alarm_active:
        cv2.rectangle(frame, (0, 0), (w, 50), (0, 0, 200), -1)
        cv2.putText(frame, "FALL DETECTED", (10, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)

    bar_x, bar_y = 10, h - 30
    bar_w, bar_h = 200, 18
    cv2.rectangle(frame, (bar_x, bar_y),
                  (bar_x + bar_w, bar_y + bar_h), (50, 50, 50), -1)
    fill = int(bar_w * float(fall_prob))
    bar_color = (0, 165, 255) if fall_prob < 0.6 else (0, 0, 255)
    cv2.rectangle(frame, (bar_x, bar_y),
                  (bar_x + fill, bar_y + bar_h), bar_color, -1)
    cv2.putText(frame, "fall {:.2f}".format(fall_prob),
                (bar_x + 5, bar_y + 14),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    cv2.putText(frame, "{:.1f} FPS".format(fps), (w - 110, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    cv2.putText(frame, "buf {}/{}".format(seq_filled, seq_len),
                (w - 110, h - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
    return frame


def run_loop(cap, save_out=None, display=True, log_every=50):
    """Main loop: read frame -> MoveNet -> features -> GRU -> alarm -> display."""
    in_w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    in_h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    in_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    print("source: {}x{} @ {:.1f} fps".format(in_w, in_h, in_fps))

    writer = None
    if save_out:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(save_out, fourcc, in_fps, (in_w, in_h))
        print("recording to: {}".format(save_out))

    feat_ext = KeypointFeatureExtractor()
    feat_ext.reset()
    feat_buffer = collections.deque(maxlen=SEQ_LEN)

    over_thr_count = 0
    alarm_remaining = 0
    fall_prob = 0.0
    frame_idx = 0

    t_last = time.time()
    fps_smooth = 0.0
    alpha_fps = 0.1

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("[INFO] stream ended at frame {}".format(frame_idx))
                break
            frame_idx += 1

            kp = movenet.infer(frame)
            feat = feat_ext.extract(kp)
            feat_buffer.append(feat)

            if (len(feat_buffer) == SEQ_LEN and
                    frame_idx % INFERENCE_STRIDE == 0):
                seq = np.stack(list(feat_buffer), axis=0)
                with torch.no_grad():
                    x = torch.from_numpy(seq).float().unsqueeze(0)
                    logits = gru(x)
                    probs = torch.softmax(logits, dim=1)[0].numpy()
                fall_prob = float(probs[1])

                if fall_prob >= PROB_THRESHOLD:
                    over_thr_count += 1
                else:
                    over_thr_count = 0
                if over_thr_count >= ALARM_FRAMES:
                    alarm_remaining = ALARM_HOLD
                    over_thr_count = 0

            if alarm_remaining > 0:
                alarm_remaining -= 1
            alarm_active = alarm_remaining > 0

            vis = draw_keypoints(frame, kp, conf_thr=DRAW_KP_THR)

            t_now = time.time()
            dt = max(t_now - t_last, 1e-6)
            cur_fps = 1.0 / dt
            t_last = t_now
            fps_smooth = ((1 - alpha_fps) * fps_smooth + alpha_fps * cur_fps
                          if fps_smooth > 0 else cur_fps)

            vis = draw_overlay(vis, fall_prob, alarm_active, fps_smooth,
                               len(feat_buffer), SEQ_LEN)

            if writer is not None:
                writer.write(vis)

            if display:
                cv2.imshow("Fall Detection", vis)
                key = cv2.waitKey(1) & 0xFF
                if key == ord("q"):
                    print("[INFO] quit by user")
                    break

            if frame_idx % log_every == 0:
                print("[frame {:5d}] fall_prob={:.2f}  alarm={}  fps={:.1f}".format(
                    frame_idx, fall_prob, alarm_active, fps_smooth))

    except KeyboardInterrupt:
        print("[INFO] interrupted")
    finally:
        cap.release()
        if writer is not None:
            writer.release()
        if display:
            cv2.destroyAllWindows()
        print("[INFO] done")

print("helpers defined")

## Run on live webcam

In [ ]:
# === Live webcam ======================================================
cap = cv2.VideoCapture(CAMERA_INDEX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  CAM_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAM_HEIGHT)
if not cap.isOpened():
    raise RuntimeError('could not open camera index {}'.format(CAMERA_INDEX))

run_loop(cap, save_out=SAVE_OUT, display=DISPLAY)

## Run on a pre-recorded video file

In [ ]:
# === Video file =======================================================
print('opening:', VIDEO_PATH)
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError('could not open video: {}'.format(VIDEO_PATH))

run_loop(cap, save_out=SAVE_OUT, display=DISPLAY)

## MoveNet-only sanity check (no GRU)

Process a single frame from the configured video. Expected: `score_max >= 0.6`.
If you see `score_max <= 0.4`, preprocessing is broken (letterbox / `/255` snuck back in).

In [ ]:
# === MoveNet sanity ==================================================
cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame = cap.read()
cap.release()
assert ret, 'failed to read a frame from {}'.format(VIDEO_PATH)

kp = movenet.infer(frame)
print('frame:', frame.shape)
print('kp shape:', kp.shape)
print('score range: [{:.3f}, {:.3f}]  mean={:.3f}'.format(
    kp[:, 2].min(), kp[:, 2].max(), kp[:, 2].mean()))

vis = draw_keypoints(frame, kp, conf_thr=0.1)
cv2.imwrite('debug_pipeline_sanity.jpg', vis)
print('saved debug_pipeline_sanity.jpg')